## Прогнозирование сердечно-сосудистых заболеваний (Framingham) логистической регрессией и KNN

Подгрузка данных

In [90]:
import pandas as pd
data = pd.read_csv("framingham.csv")

In [91]:
data

,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,4.0,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,0,46,2.0,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,1,48,1.0,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,0,61,3.0,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,0,46,3.0,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4233,1,50,1.0,1,1.0,0.0,0,1,0,313.0,179.0,92.0,25.97,66.0,86.0,1
4234,1,51,3.0,1,43.0,0.0,0,0,0,207.0,126.5,80.0,19.71,65.0,68.0,0
4235,0,48,2.0,1,20.0,NaN,0,0,0,248.0,131.0,72.0,22.00,84.0,86.0,0
4236,0,44,1.0,1,15.0,0.0,0,0,0,210.0,126.5,87.0,19.16,86.0,NaN,0


Проверка пропущенных значений 

In [92]:
missing_counts = data.isnull().sum()

print(missing_counts)

male                 0
age                  0
education          105
currentSmoker        0
cigsPerDay          29
BPMeds              53
prevalentStroke      0
prevalentHyp         0
diabetes             0
totChol             50
sysBP                0
diaBP                0
BMI                 19
heartRate            1
glucose            388
TenYearCHD           0
dtype: int64


Разделение на результативный и факторные признаки

In [93]:
X = data.drop(columns=["TenYearCHD"])
y = data["TenYearCHD"]

In [94]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

Разделение на тестовую и обучающую выборки и заполнение пропущенных значений

In [95]:
num_features = X.select_dtypes(include="number").columns

preproc = ColumnTransformer([
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler())
    ]), num_features)
], remainder="drop")

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

Модель 1 - Логистическая регрессия ( построение и обучение )

In [96]:
from sklearn.linear_model import LogisticRegression
clf_log = Pipeline([
    ("prep", preproc),
    ("clf", LogisticRegression(max_iter=1000, solver="lbfgs"))
])
clf_log.fit(X_train, y_train)
y_pred_log = clf_log.predict(X_test)

Модель 2 - Метод к-ближайших соседей ( построение и обучение )

In [97]:
from sklearn.neighbors import KNeighborsClassifier

In [98]:
from sklearn.model_selection import GridSearchCV

knn_pipe = Pipeline([("prep", preproc),
                     ("clf", KNeighborsClassifier())])

param_grid = {"clf__n_neighbors": [3,5,7,9,11,15]}
grid = GridSearchCV(knn_pipe, param_grid=param_grid,
                    scoring="accuracy", cv=5, n_jobs=-1)
grid.fit(X_train, y_train)

best_knn = grid.best_estimator_
y_pred_knn = best_knn.predict(X_test)
print("Best k:", grid.best_params_["clf__n_neighbors"])

Best k: 11


In [99]:
clf_knn = Pipeline([
    ("prep", preproc),
    ("clf", KNeighborsClassifier(n_neighbors=11))
])
clf_knn.fit(X_train, y_train)
y_pred_knn = clf_knn.predict(X_test)

In [100]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

# ЛОГИСТИЧЕСКАЯ РЕГРЕССИЯ
acc_log = accuracy_score(y_test, y_pred_log)
prec_log = precision_score(y_test, y_pred_log, zero_division=0)
rec_log = recall_score(y_test, y_pred_log, zero_division=0)
print("Accuracy:", acc_log)
print("Precision:", prec_log)
print("Recall:", rec_log)

# KNN
acc_knn = accuracy_score(y_test, y_pred_knn)
prec_knn = precision_score(y_test, y_pred_knn, zero_division=0)
rec_knn = recall_score(y_test, y_pred_knn, zero_division=0)
print("Accuracy:", acc_knn)
print("Precision:", prec_knn)
print("Recall:", rec_knn)

Accuracy: 0.8474842767295597
Precision: 0.48148148148148145
Recall: 0.06735751295336788
Accuracy: 0.8474842767295597
Precision: 0.4782608695652174
Recall: 0.05699481865284974


In [101]:
from sklearn.metrics import confusion_matrix, classification_report
print(confusion_matrix(y_test, y_pred_log))
print(classification_report(y_test, y_pred_log))

[[1065   14]
 [ 180   13]]
              precision    recall  f1-score   support

           0       0.86      0.99      0.92      1079
           1       0.48      0.07      0.12       193

    accuracy                           0.85      1272
   macro avg       0.67      0.53      0.52      1272
weighted avg       0.80      0.85      0.80      1272



In [102]:
import pandas as pd

results = pd.DataFrame({
    "Model": ["LogisticRegression", "KNN"],
    "Accuracy": [acc_log, acc_knn],
    "Precision": [prec_log, prec_knn],
    "Recall": [rec_log, rec_knn]
})
print(results)

                Model  Accuracy  Precision    Recall
0  LogisticRegression  0.847484   0.481481  0.067358
1                 KNN  0.847484   0.478261  0.056995


ПОСТРОЕНИЕ МОДЕЛЕЙ ПО ОГРАНИЧЕННОМУ КОЛИЧЕСТВУ РЕЗУЛЬТАТИВНЫХ ПРИЗНАКОВ 

In [103]:
selected_features = ["age", "male", "currentSmoker", "cigsPerDay", "diabetes", "diaBP", "sysBP", "glucose"]
z = data[selected_features]
g = data["TenYearCHD"]

Разделение на тестовую и обучающую выборки и заполнение пропущенных значений

In [104]:
num_features = z.select_dtypes(include="number").columns

preproc = ColumnTransformer([
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler())
    ]), num_features)
], remainder="drop")

from sklearn.model_selection import train_test_split
z_train, z_test, g_train, g_test = train_test_split(
    z, g, test_size=0.30, random_state=42, stratify=g
)

Логистическая регрессия с ограниченным колиством факторов

In [105]:
from sklearn.linear_model import LogisticRegression
clf_log = Pipeline([
    ("prep", preproc),
    ("clf", LogisticRegression(max_iter=1000, solver="lbfgs"))
])
clf_log.fit(z_train, g_train)
g_pred_log = clf_log.predict(z_test)

Модель 2 - Метод к-ближайших соседей с ограниченным количеством факторов 

In [106]:
from sklearn.model_selection import GridSearchCV

knn_pipe = Pipeline([("prep", preproc),
                     ("clf", KNeighborsClassifier())])

param_grid = {"clf__n_neighbors": [3,5,7,9,11,15]}
grid = GridSearchCV(knn_pipe, param_grid=param_grid,
                    scoring="accuracy", cv=5, n_jobs=-1)
grid.fit(z_train, g_train)

best_knn = grid.best_estimator_
g_pred_knn = best_knn.predict(z_test)
print("Best k:", grid.best_params_["clf__n_neighbors"])

Best k: 15


In [107]:
clf_knn = Pipeline([
    ("prep", preproc),
    ("clf", KNeighborsClassifier(n_neighbors=15))
])
clf_knn.fit(z_train, g_train)
g_pred_knn = clf_knn.predict(z_test)

In [108]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

# ЛОГИСТИЧЕСКАЯ РЕГРЕССИЯ
acc_log = accuracy_score(g_test, g_pred_log)
prec_log = precision_score(g_test, g_pred_log, zero_division=0)
rec_log = recall_score(g_test, g_pred_log, zero_division=0)
print("Accuracy:", acc_log)
print("Precision:", prec_log)
print("Recall:", rec_log)

# KNN
acc_knn = accuracy_score(g_test, g_pred_knn)
prec_knn = precision_score(g_test, g_pred_knn, zero_division=0)
rec_knn = recall_score(g_test, g_pred_knn, zero_division=0)
print("Accuracy:", acc_knn)
print("Precision:", prec_knn)
print("Recall:", rec_knn)

Accuracy: 0.845125786163522
Precision: 0.4230769230769231
Recall: 0.05699481865284974
Accuracy: 0.8435534591194969
Precision: 0.39285714285714285
Recall: 0.05699481865284974


In [109]:
from sklearn.metrics import confusion_matrix, classification_report
print(confusion_matrix(g_test, g_pred_log))
print(classification_report(g_test, g_pred_log))

[[1064   15]
 [ 182   11]]
              precision    recall  f1-score   support

           0       0.85      0.99      0.92      1079
           1       0.42      0.06      0.10       193

    accuracy                           0.85      1272
   macro avg       0.64      0.52      0.51      1272
weighted avg       0.79      0.85      0.79      1272



In [110]:
import pandas as pd

results = pd.DataFrame({
    "Model": ["LogisticRegression", "KNN"],
    "Accuracy": [acc_log, acc_knn],
    "Precision": [prec_log, prec_knn],
    "Recall": [rec_log, rec_knn]
})
print(results)Accuracy

                Model  Accuracy  Precision    Recall
0  LogisticRegression  0.845126   0.423077  0.056995
1                 KNN  0.843553   0.392857  0.056995


Вывод: 
Для начала в работе были подгружены данные, после чего на этапе предобработки были рассмотрены значения имеющие пропуски. Мы выяснили, что пропуски составляют более 10% значений, соответственно удаление пропусков является невозможным. Соответсвенно пропуски были заполнены медианными значениями, как в случае построение регрессионных моделей по всем признакам, так и в случае ограниченного количества факторных признаков. Как в первом, rак и во втором случае мы разделелили данные на результативный признак и факторные. В первом случае факторными признаками стали все признаки, кроме TenYearCHD - результативного. Во втором случае результативным также остался TenYearCHD, а факторные были выбраны следующие: "age", "male", "currentSmoker", "cigsPerDay", "diabetes", "diaBP", "sysBP", "glucose". После чего в обоих случаях данные были разделены на обучающую и тестовую выборки и были построены модели с помощью LogisticRegression и метода к-ближайших соседей. При сравнении результатов в приведенных выше " сводных таблицах ", отражающих значения одинаковых метрик: Accuracy, Precision, Recall, по каждой из модели и в первом и во втором случае, было выявленно: в случае построения моделей регрессии, включая все признаки, как факторы лучшей моделью для описания результативного признака является  LogisticRegression, на основе того, что значения метрик Precision, Recall выше, чем у модели построенной методом KNN; в случае построение модели с ограниченным числом факторов (8), наилучшаа модель также была построенна методом LogisticRegression. В общем при сравнении моделей со всеми признаками - факторами и ограниченным числом факторов, можно отметить, что значения метрик, в случае ограниченного количества факторных признаков, меньше, что говорит о том, что модели регрессии построенные по всем признакам, кроме результативного, лучше подходят для описания y - в первом случае и g - во втором случае ( результативные признаки). Таким образом построив 4 модели регрессии двумя разными методами по разному количеству факторных признаков можно сделать вывод о том, что наилучшей моделью является модель построенная по всем признакам - факторам (15), кроме результативного TenYearCHD, методом LogisticRegression, с метриками равными: Accuracy - 0.847484, Precision - 0.481481, Recall - 0.067358.